In [9]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

m,cm=1,0.01
class Detector():
    def __init__(self, R=2*m,N=40,r=10*cm, prob=0.5):
        self.R=R
        self.N=N
        self.r=r
        self.prob=prob
        self.PMTArray=self.uniform_distribution()
    def uniform_distribution(self):
        i = np.arange(self.N)
        golden_ratio = (1 + np.sqrt(5)) / 2
        cosTheta = 1 - 2 * (i + 0.5) / self.N
        sinTheta = np.sqrt(1 - cosTheta*cosTheta)
        phi = 2 * np.pi * i / golden_ratio
        return self.R * np.array([sinTheta * np.cos(phi),sinTheta * np.sin(phi),cosTheta])
        
    def IsInPMT(self,origin:np.ndarray,PhotonArray:np.ndarray):
        cosMax=np.cos(self.r/self.R)
        cosCheck=(origin[:, None]+PhotonArray).T@ self.PMTArray/ (self.R ** 2)
        return np.any(cosCheck>cosMax, axis=1)
        
    def ElectronYield(self, NDetected:float):
        return np.random.poisson(self.prob * NDetected)
        
    #Все методы изначально были построены преимущественно под numpy:array, поэтому ниже будет игрушечное сравнение с методами, подстроенными под list 
    #аналог метода IsInPMT для сравнения по времени выполнения
    def in_pmt_one(self, origin, hit, PMT, cosMax, R):
        cosCheck = (origin + hit) @ PMT / (R**2) 
        return np.any(cosCheck > cosMax)

class Source():
    def __init__(self, E:float,origin:np.ndarray,Y=1e4,L=8):
        self.Y=Y
        self.L=L
        self.N=np.random.poisson(E*Y) 
        self.origin=origin
    def photon_generator(self, bunch=100):
        left = self.N
        while left > 0:
            n = min(bunch, left)
            phi = np.random.uniform(0, 2 * np.pi, n)
            cosTheta = np.random.uniform(-1, 1, n)
            sinTheta = np.sqrt(1 - cosTheta * cosTheta)
            yield np.array([sinTheta * np.cos(phi),sinTheta * np.sin(phi),cosTheta]) 
            left -= n
            
    def random_distribution(self):
        phi=np.random.uniform(0,2*np.pi, self.N)
        cosTheta=np.random.uniform(-1,1,self.N)
        sinTheta=np.sqrt(1-cosTheta*cosTheta)
        return np.array([sinTheta*np.cos(phi),sinTheta*np.sin(phi), cosTheta])
        
    def distance_traversed(self,PhotonArray:np.ndarray,R:float):
        t=-self.origin@PhotonArray+np.sqrt(R**2-np.sum(self.origin**2)+(self.origin@PhotonArray)*(self.origin@PhotonArray))
        return t
    def attenuation(self, PhotonArray, R):
        t = self.distance_traversed(PhotonArray, R)
        gen_dist = -self.L*np.log(np.random.uniform(0, 1, len(t))) 
        return t<gen_dist
        

    #аналоги методов для сравнения по времени выполнения 
    
    def photon_one(self):
        for _ in range(self.N):
            phi = np.random.uniform(0, 2 * np.pi)
            cosTheta = np.random.uniform(-1, 1)
            sinTheta = np.sqrt(1 - cosTheta * cosTheta)
            yield np.array([sinTheta * np.cos(phi),sinTheta * np.sin(phi),cosTheta])
    def distance_one(self, u, R):
        return -self.origin @ u + np.sqrt(R**2 - np.sum(self.origin**2) + (self.origin @ u)**2)
    



In [10]:
D1=Detector()
origin=np.array([0,0,0])
S1=Source(1,origin)

def event_list(S1, D1, attenuate=True):
    registered = 0
    cosMax = np.cos(D1.r / D1.R)
    for u in S1.photon_one():
        t = S1.distance_one(u, D1.R)
        if attenuate and not (t < -S1.L * np.log(np.random.random())):
            continue
        if D1.in_pmt_one(S1.origin, u * t, D1.PMTArray, cosMax, D1.R):
            registered += 1
    return D1.ElectronYield(registered)

def event_numpy(S1, D1, attenuate=True):
    u = S1.random_distribution()
    t = S1.distance_traversed(u, D1.R)
    hits = u * t
    if attenuate:
        hits = hits[:, S1.attenuation(u,D1.R)]
    registered = np.sum(D1.IsInPMT(S1.origin, hits))
    return D1.ElectronYield(registered)

In [12]:
import time

n_events = 50
t0 = time.perf_counter()

for _ in range(n_events):
    event_list(S1, D1)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(n_events):
    event_numpy(S1, D1)
    
t_np = time.perf_counter() - t0
print(f'T_list={t_loop} T_np={t_np}\nбуст по времени = {t_loop / t_np}')

T_list=20.29992069999571 T_np=0.23595170000044163
буст по времени = 86.03422098657359
